# ECG AF Benchmark Framework Demo

This notebook runs the new `py_ecg_benchmark_pkg` package from source and demonstrates the benchmark pieces that do not require live infrastructure: config loading, artifact layout, deterministic splits, window labeling, metrics, and baseline model behavior.

The final section follows the working processed-data EDA notebook pattern: list MinIO/S3 keys with `boto3`, read parquet and annotation JSON through Spark `s3a://`, then run a small baseline model over real annotation-derived features.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "py_ecg_benchmark_pkg").exists():
    REPO_ROOT = Path.cwd().parent

PKG_SRC = REPO_ROOT / "py_ecg_benchmark_pkg" / "src"
if str(PKG_SRC) not in sys.path:
    sys.path.insert(0, str(PKG_SRC))

EXAMPLE_CONFIG = REPO_ROOT / "py_ecg_benchmark_pkg" / "examples" / "benchmark.yaml"

required_demo_packages = {
    "yaml": "PyYAML>=6.0",
    "boto3": "boto3>=1.34",
}
for module_name, package_spec in required_demo_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_spec} into this notebook kernel...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_spec])

print(f"repo: {REPO_ROOT}")
print(f"package source: {PKG_SRC}")
print(f"example config: {EXAMPLE_CONFIG}")

## Load And Inspect The Example Config

In [ ]:
from ecg_benchmark.config import load_config
from ecg_benchmark.artifacts import ArtifactLayout, make_run_id

config = load_config(EXAMPLE_CONFIG, validate=True)
run_id = make_run_id(config)
layout = ArtifactLayout.from_config(config, run_id)

print("experiment:", config.experiment.name)
print("random_seed:", config.random_seed)
print("config_hash:", config.config_hash())
print("source_datasets:", config.data.source_datasets)
print("target_datasets:", config.data.target_datasets)
print("artifact root:", layout.run_uri)

In [ ]:
artifact_examples = {
    "resolved_config": layout.resolved_config_uri(),
    "split_manifest": layout.split_manifest_uri(),
    "feature_dataset": layout.feature_dataset_uri(config.data.source_datasets[0]),
    "model": layout.model_uri(config.data.source_datasets[0], "dummy_prior"),
    "metrics_json": layout.metrics_json_uri(),
    "matrix_csv": layout.matrix_csv_uri(),
}
artifact_examples

## Deterministic Subject-Safe Splits

In [ ]:
from ecg_benchmark.splits import RecordingRecord, make_deterministic_splits, split_manifest

records = [
    RecordingRecord("demo-dataset", "rec-001", subject_id="subj-1", af_present=True, duration_seconds=600),
    RecordingRecord("demo-dataset", "rec-002", subject_id="subj-1", af_present=True, duration_seconds=640),
    RecordingRecord("demo-dataset", "rec-003", subject_id="subj-2", af_present=False, duration_seconds=500),
    RecordingRecord("demo-dataset", "rec-004", subject_id="subj-3", af_present=False, duration_seconds=520),
    RecordingRecord("demo-dataset", "rec-005", subject_id=None, af_present=True, duration_seconds=480),
    RecordingRecord("demo-dataset", "rec-006", subject_id=None, af_present=False, duration_seconds=450),
]

assignments = make_deterministic_splits(
    records,
    seed=config.random_seed,
    train_ratio=config.data.split.train,
    val_ratio=config.data.split.val,
    test_ratio=config.data.split.test,
    stratify_by=config.data.split.stratify_by,
)

manifest = split_manifest(
    assignments,
    seed=config.random_seed,
    config_hash=config.config_hash(),
    ratios={"train": config.data.split.train, "val": config.data.split.val, "test": config.data.split.test},
)

print("split_hash:", manifest["split_hash"])
print("counts:", manifest["counts"])
manifest["assignments"]

## AF-vs-NSR Window Labels

In [ ]:
from ecg_benchmark.labels import episodes_from_annotation_json, labels_for_recording

annotation_sidecar = {
    "target_sampling_rate": 250,
    "rhythm_episodes": [
        {"start_sample": 0, "end_sample": 2500, "rhythm_type": "NSR"},
        {"start_sample": 2500, "end_sample": 5000, "rhythm_type": "AF"},
        {"start_sample": 5000, "end_sample": 7500, "rhythm_type": "AFL"},
    ],
}

episodes = episodes_from_annotation_json(annotation_sidecar)
window_labels = labels_for_recording(
    duration_seconds=30,
    episodes=episodes,
    window_seconds=config.data.window_seconds,
    stride_seconds=config.data.stride_seconds,
    policy=config.data.label_policy,
)

[item for item in window_labels]

The default policy keeps only windows fully inside `AF` or `NSR`. Mixed-boundary windows and `AFL` windows are dropped by assigning `label=None`.

## Baseline Features, Models, And Metrics

In [ ]:
from ecg_benchmark.features import numeric_window_features
from ecg_benchmark.metrics import BinaryPrediction, aggregate_by_recording, compute_binary_metrics
from ecg_benchmark.models import create_model, model_infos

for info in model_infos():
    print(f"{info.model_id}: {info.description}")

In [ ]:
feature_rows = [
    {"full_recording_id": "rec-a", "label": 1, "rr_cv": 0.28, **numeric_window_features([0.2, 0.4, 0.1], rr_intervals_seconds=[0.7, 1.1, 0.6])},
    {"full_recording_id": "rec-a", "label": 1, "rr_cv": 0.31, **numeric_window_features([0.3, 0.5, 0.2], rr_intervals_seconds=[0.8, 1.2, 0.7])},
    {"full_recording_id": "rec-b", "label": 0, "rr_cv": 0.04, **numeric_window_features([0.1, 0.1, 0.2], rr_intervals_seconds=[0.9, 0.92, 0.88])},
    {"full_recording_id": "rec-b", "label": 0, "rr_cv": 0.05, **numeric_window_features([0.2, 0.1, 0.2], rr_intervals_seconds=[0.95, 0.91, 0.93])},
]

dummy = create_model("dummy_prior").fit(feature_rows, [], {})
rr_rule = create_model("rr_irregularity_rule").fit(feature_rows, [], {"threshold": 0.12, "slope": 12.0})

dummy_scores = dummy.predict_proba(feature_rows)
rr_scores = rr_rule.predict_proba(feature_rows)

print("dummy_prior scores:", [round(x, 3) for x in dummy_scores])
print("rr_irregularity_rule scores:", [round(x, 3) for x in rr_scores])

In [ ]:
window_predictions = [
    BinaryPrediction(y_true=int(row["label"]), y_score=score, recording_id=row["full_recording_id"])
    for row, score in zip(feature_rows, rr_scores)
]

recording_predictions = aggregate_by_recording(window_predictions, policy=config.data.recording_aggregation)
metrics = compute_binary_metrics(recording_predictions, metric_names=config.metrics)

print("recording predictions:")
for item in recording_predictions:
    print(item)

metrics

## CLI Smoke Test

In [ ]:
from ecg_benchmark.cli import main as ecg_bench_main

exit_code = ecg_bench_main([
    "validate-config",
    "--config", str(EXAMPLE_CONFIG),
    "--skip-env",
])
print("exit_code:", exit_code)

## Real Processed Data: MinIO/S3 + Spark

This section follows the same data-loading pattern as `experiments/data_pipeline_test/01_fetch_and_eda_processed_250hz.ipynb`: use `boto3` to list processed files and Spark `s3a://` to read parquet/annotation JSON directly from MinIO. It does not require Postgres benchmark tables.

When this notebook runs inside the `spark-eda` container, `S3_ENDPOINT_URL=http://minio:9000` is expected. When it runs on the host, the equivalent endpoint is usually `http://localhost:9000`. The listing cell below handles both cases.

The goal here is not a final benchmark score yet. It is a real-data smoke test that verifies we can discover processed files, read ECG parquet and annotation sidecars, convert rhythm episodes into AF/NSR windows, build simple RR-interval features, and run the baseline model API on those features.


In [ ]:
from collections import defaultdict
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("boto3") is None:
    print("Installing boto3 into this notebook kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3>=1.34"])

import boto3
from botocore.config import Config
from botocore.exceptions import EndpointConnectionError


def load_dotenv_values(path):
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


pipeline_env = load_dotenv_values(REPO_ROOT / "data-pipeline" / ".env")


def env_or_file(name, default=None):
    return os.getenv(name) or pipeline_env.get(name) or default


S3_ENDPOINT_URL = env_or_file("S3_ENDPOINT_URL", "http://localhost:9000")
S3_ACCESS_KEY = env_or_file("S3_ACCESS_KEY", "minioadmin")
S3_SECRET_KEY = env_or_file("S3_SECRET_KEY", "minioadmin")
S3_BUCKET = env_or_file("S3_BUCKET", config.storage.source_bucket)
PREFIX = f"{config.data.processed_prefix}/{config.data.sample_rate_hz}hz/"


def endpoint_candidates(endpoint):
    candidates = [endpoint]
    if "://minio" in endpoint:
        candidates.append(endpoint.replace("://minio", "://localhost"))
        candidates.append(endpoint.replace("://minio", "://127.0.0.1"))
    elif "://localhost" in endpoint:
        candidates.append(endpoint.replace("://localhost", "://127.0.0.1"))
    seen = set()
    return [item for item in candidates if not (item in seen or seen.add(item))]


def make_s3_client(endpoint):
    return boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=S3_ACCESS_KEY,
        aws_secret_access_key=S3_SECRET_KEY,
        region_name="us-east-1",
        config=Config(s3={"addressing_style": "path"}),
    )


last_error = None
for endpoint in endpoint_candidates(S3_ENDPOINT_URL):
    parquet_keys = []
    annotation_keys = []
    keys_by_dataset = defaultdict(list)
    annotation_keys_by_dataset = defaultdict(list)
    try:
        s3 = make_s3_client(endpoint)
        paginator = s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=PREFIX):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                relative_key = key[len(PREFIX):] if key.startswith(PREFIX) else key
                dataset_name = relative_key.split("/", 1)[0]
                if key.endswith(".parquet"):
                    parquet_keys.append(key)
                    keys_by_dataset[dataset_name].append(key)
                elif key.endswith(".annotations.json"):
                    annotation_keys.append(key)
                    annotation_keys_by_dataset[dataset_name].append(key)
        S3_ENDPOINT_URL = endpoint
        print(f"Connected to MinIO/S3 at {S3_ENDPOINT_URL}")
        break
    except EndpointConnectionError as exc:
        last_error = exc
        print(f"Could not connect to {endpoint}; trying next endpoint if available.")
else:
    tried = ", ".join(endpoint_candidates(S3_ENDPOINT_URL))
    raise RuntimeError(
        f"Could not connect to MinIO/S3. Tried: {tried}. "
        "If the notebook runs on the host, use S3_ENDPOINT_URL=http://localhost:9000; "
        "inside Docker, use S3_ENDPOINT_URL=http://minio:9000."
    ) from last_error

print(f"Found {len(parquet_keys)} parquet files in s3://{S3_BUCKET}/{PREFIX}")
print(f"Found {len(annotation_keys)} annotation JSON files in s3://{S3_BUCKET}/{PREFIX}")
print(f"Found {len(keys_by_dataset)} datasets under {PREFIX}")
for dataset_name in sorted(keys_by_dataset):
    print(
        f" - {dataset_name}: {len(keys_by_dataset[dataset_name])} parquet files, "
        f"{len(annotation_keys_by_dataset.get(dataset_name, []))} annotation JSON files"
    )


In [ ]:
if not parquet_keys:
    raise RuntimeError(f"No parquet files found under s3://{S3_BUCKET}/{PREFIX}. Run the pipeline first.")

from urllib.parse import urlparse

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

preferred_datasets = config.data.source_datasets + config.data.target_datasets
datasets_with_annotations = {name for name, keys in annotation_keys_by_dataset.items() if keys}
dataset_name = next(
    (name for name in preferred_datasets if name in keys_by_dataset and name in datasets_with_annotations),
    next((name for name in sorted(keys_by_dataset) if name in datasets_with_annotations), sorted(keys_by_dataset)[0]),
)
dataset_root = f"s3a://{S3_BUCKET}/{PREFIX}{dataset_name}/"
annotation_paths = [f"s3a://{S3_BUCKET}/{key}" for key in annotation_keys_by_dataset.get(dataset_name, [])]

parsed_endpoint = urlparse(S3_ENDPOINT_URL)
s3a_endpoint = parsed_endpoint.netloc or parsed_endpoint.path
s3a_ssl_enabled = parsed_endpoint.scheme == "https"

spark = (
    SparkSession.builder
    .master(os.getenv("SPARK_MASTER", "local[*]"))
    .appName("af-benchmark-framework-demo")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", S3_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(s3a_ssl_enabled).lower())
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

recording_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.parquet")
    .parquet(dataset_root)
)

print(f"Selected dataset: {dataset_name}")
print(f"Dataset root: {dataset_root}")
recording_df.printSchema()
recording_df.show(5, truncate=False)

if not annotation_paths:
    raise RuntimeError(f"No annotation JSON files found for {dataset_name}; choose a dataset with rhythm annotations.")

annotations_df = spark.read.option("multiLine", "true").json(annotation_paths)
annotations_df.printSchema()

if set(annotations_df.columns) == {"_corrupt_record"}:
    raise RuntimeError(
        f"Annotation JSON could not be parsed for {dataset_name}. "
        "This matches Spark's _corrupt_record fallback; inspect annotation_paths[:3]."
    )

for required_column in ["recording_id", "target_sampling_rate", "rhythm_episodes"]:
    if required_column not in annotations_df.columns:
        raise RuntimeError(f"Annotation JSON for {dataset_name} is missing required column: {required_column}")

preview_columns = [column for column in ["recording_id", "target_sampling_rate", "beat_annotations", "rhythm_episodes"] if column in annotations_df.columns]
annotations_df.select(*preview_columns).show(5, truncate=80)

if "beat_annotations" in annotations_df.columns:
    beat_df = (
        annotations_df
        .where(F.col("beat_annotations").isNotNull())
        .where(F.col("beat_annotations.positions").isNotNull())
        .select(
            "recording_id",
            "target_sampling_rate",
            F.posexplode("beat_annotations.positions").alias("beat_number", "beat_sample_index"),
            F.col("beat_annotations.labels").alias("_beat_labels"),
        )
        .withColumn("beat_label", F.element_at(F.col("_beat_labels"), F.col("beat_number") + F.lit(1)))
        .drop("_beat_labels")
    )
else:
    beat_df = spark.createDataFrame(
        [],
        "recording_id string, target_sampling_rate int, beat_number int, beat_sample_index int, beat_label string",
    )

rhythm_df = (
    annotations_df
    .where(F.col("rhythm_episodes").isNotNull())
    .select("recording_id", "target_sampling_rate", F.explode("rhythm_episodes").alias("episode"))
    .select(
        "recording_id",
        "target_sampling_rate",
        F.col("episode.start_sample").alias("start_sample"),
        F.col("episode.end_sample").alias("end_sample"),
        F.col("episode.rhythm_type").alias("rhythm_type"),
    )
)

print("Beat annotations:")
beat_df.show(10, truncate=False)
print("Rhythm episodes:")
rhythm_df.show(10, truncate=False)

## Real-Data Baseline Interpretation

The previous cell confirms that Spark can read the selected processed dataset and annotation JSON sidecars. For the current run it selected `mit-bih-atrial-fibrillation-database`, read sample-level ECG columns such as `sample_index`, `channel_0_ECG1`, and `channel_1_ECG2`, and flattened annotation sidecars into beat positions and rhythm episodes.

The baseline below uses only annotation-derived RR interval features. It does not inspect ECG waveform morphology yet. Windows are labeled with the benchmark policy: keep only windows fully inside `AF` or `NSR`; drop mixed-boundary, `OTHER`, `UNKNOWN`, and `AFL` windows.

Earlier demo versions used row-order or window-level splits. Both can be misleading: row-order splits can leave the test set with only one class, and window-level splits can leak recording-specific rhythm patterns by putting windows from the same recording in both train and test.

The next cell samples a bounded number of windows per recording, then performs a deterministic recording-disjoint split. Whole recordings go either to train or test. This is closer to the benchmark goal because the model is evaluated on recordings it did not see during fitting. It is still a notebook smoke test, not the final subject-safe split from the full benchmark workflow.

The cell reports two views of performance. Window-level metrics score each 10-second window directly. Recording-level metrics first aggregate scores per held-out recording using the configured aggregation policy; the recording truth is positive if any evaluated window in that recording is AF. If the held-out recordings contain only AF-present recordings, some recording-level metrics can still be `None`; that is an honest limitation of the small demo sample.


In [ ]:
from collections import defaultdict
import importlib
import random

import ecg_benchmark.metrics as metrics_mod
from ecg_benchmark.features import rr_interval_features
from ecg_benchmark.labels import episodes_from_annotation_json, labels_for_recording
from ecg_benchmark.models import create_model

# The notebook kernel may have imported ecg_benchmark.metrics before package edits.
# Reload it so rerunning this cell uses the current recording aggregation semantics:
# a recording is positive if any evaluated window is AF.
metrics_mod = importlib.reload(metrics_mod)
BinaryPrediction = metrics_mod.BinaryPrediction
aggregate_by_recording = metrics_mod.aggregate_by_recording
compute_binary_metrics = metrics_mod.compute_binary_metrics

feature_source_columns = [
    column
    for column in ["recording_id", "target_sampling_rate", "beat_annotations", "rhythm_episodes"]
    if column in annotations_df.columns
]

MAX_RECORDINGS = 50
MAX_WINDOWS_PER_RECORDING = 80
MAX_TOTAL_WINDOWS = 1500

real_feature_rows = []
recordings_with_features = 0
for row in annotations_df.select(*feature_source_columns).limit(MAX_RECORDINGS).collect():
    payload = row.asDict(recursive=True)
    if not payload.get("rhythm_episodes"):
        continue
    rate = int(payload.get("target_sampling_rate") or config.data.sample_rate_hz)
    episodes = episodes_from_annotation_json(payload, sample_rate_hz=rate)
    if not episodes:
        continue
    duration_seconds = max(ep.end_seconds for ep in episodes)
    beat_positions = ((payload.get("beat_annotations") or {}).get("positions") or [])

    rows_for_recording = []
    for window in labels_for_recording(
        duration_seconds=duration_seconds,
        episodes=episodes,
        window_seconds=config.data.window_seconds,
        stride_seconds=config.data.stride_seconds,
        policy=config.data.label_policy,
    ):
        if window.label is None:
            continue
        start_sample = int(round(window.start_seconds * rate))
        end_sample = int(round(window.end_seconds * rate))
        positions = sorted(int(pos) for pos in beat_positions if start_sample <= int(pos) < end_sample)
        intervals = [(right - left) / float(rate) for left, right in zip(positions, positions[1:]) if right > left]
        rr_features = rr_interval_features(intervals)
        if rr_features["rr_count"] < 2:
            continue
        rows_for_recording.append({
            "full_recording_id": payload["recording_id"],
            "label": int(window.label),
            "window_start_seconds": window.start_seconds,
            "window_end_seconds": window.end_seconds,
            **rr_features,
        })
        if len(rows_for_recording) >= MAX_WINDOWS_PER_RECORDING:
            break

    if rows_for_recording:
        recordings_with_features += 1
        real_feature_rows.extend(rows_for_recording)
    if len(real_feature_rows) >= MAX_TOTAL_WINDOWS:
        real_feature_rows = real_feature_rows[:MAX_TOTAL_WINDOWS]
        break

print(f"Built {len(real_feature_rows)} labeled RR feature windows from {recordings_with_features} recordings.")
label_counts = {label: sum(1 for row in real_feature_rows if row["label"] == label) for label in [0, 1]}
print("window label counts:", label_counts)


def recording_disjoint_split(rows, *, seed, test_fraction=0.30):
    by_recording = defaultdict(list)
    for item in rows:
        by_recording[item["full_recording_id"]].append(item)

    recording_label = {
        recording_id: 1 if any(row["label"] == 1 for row in recording_rows) else 0
        for recording_id, recording_rows in by_recording.items()
    }
    by_label = defaultdict(list)
    for recording_id, label in sorted(recording_label.items()):
        by_label[label].append(recording_id)

    rng = random.Random(seed)
    train_recordings = set()
    test_recordings = set()
    for label, recording_ids in sorted(by_label.items()):
        shuffled = list(recording_ids)
        rng.shuffle(shuffled)
        if len(shuffled) == 1:
            train_recordings.add(shuffled[0])
            continue
        test_count = max(1, int(round(len(shuffled) * test_fraction)))
        test_count = min(test_count, len(shuffled) - 1)
        test_recordings.update(shuffled[:test_count])
        train_recordings.update(shuffled[test_count:])

    if not test_recordings and len(train_recordings) > 1:
        moved = sorted(train_recordings)[-1]
        train_recordings.remove(moved)
        test_recordings.add(moved)

    train_rows = [row for row in rows if row["full_recording_id"] in train_recordings]
    test_rows = [row for row in rows if row["full_recording_id"] in test_recordings]
    return train_rows, test_rows, recording_label, train_recordings, test_recordings


if len(real_feature_rows) >= 4 and all(label_counts.values()):
    train_rows, test_rows, recording_label, train_recordings, test_recordings = recording_disjoint_split(
        real_feature_rows,
        seed=config.random_seed,
        test_fraction=0.30,
    )

    train_counts = {label: sum(1 for row in train_rows if row["label"] == label) for label in [0, 1]}
    test_counts = {label: sum(1 for row in test_rows if row["label"] == label) for label in [0, 1]}
    train_recording_counts = {label: sum(1 for rid in train_recordings if recording_label[rid] == label) for label in [0, 1]}
    test_recording_counts = {label: sum(1 for rid in test_recordings if recording_label[rid] == label) for label in [0, 1]}

    print("train window label counts:", train_counts)
    print("test window label counts:", test_counts)
    print("train recording counts by AF-present label:", train_recording_counts)
    print("test recording counts by AF-present label:", test_recording_counts)
    print("recording overlap between train and test:", sorted(train_recordings & test_recordings))

    if not train_rows or not test_rows:
        raise RuntimeError("Recording-disjoint split did not produce both train and test rows. Increase MAX_RECORDINGS.")

    real_dummy = create_model("dummy_prior").fit(train_rows, [], {})
    real_rr_rule = create_model("rr_irregularity_rule").fit(train_rows, [], {"threshold": 0.12, "slope": 12.0})

    dummy_scores = real_dummy.predict_proba(test_rows)
    rr_scores = real_rr_rule.predict_proba(test_rows)

    dummy_predictions = [
        BinaryPrediction(y_true=int(row["label"]), y_score=float(score), recording_id=row["full_recording_id"])
        for row, score in zip(test_rows, dummy_scores)
    ]
    rr_predictions = [
        BinaryPrediction(y_true=int(row["label"]), y_score=float(score), recording_id=row["full_recording_id"])
        for row, score in zip(test_rows, rr_scores)
    ]

    dummy_window_metrics = compute_binary_metrics(dummy_predictions, metric_names=config.metrics)
    rr_window_metrics = compute_binary_metrics(rr_predictions, metric_names=config.metrics)

    dummy_recording_predictions = aggregate_by_recording(dummy_predictions, policy=config.data.recording_aggregation)
    rr_recording_predictions = aggregate_by_recording(rr_predictions, policy=config.data.recording_aggregation)

    dummy_recording_metrics = compute_binary_metrics(dummy_recording_predictions, metric_names=config.metrics)
    rr_recording_metrics = compute_binary_metrics(rr_recording_predictions, metric_names=config.metrics)

    print("dummy_prior window-level metrics:")
    print(dummy_window_metrics)
    print("rr_irregularity_rule window-level metrics:")
    print(rr_window_metrics)
    print("dummy_prior recording-level metrics:")
    print(dummy_recording_metrics)
    print("rr_irregularity_rule recording-level metrics:")
    print(rr_recording_metrics)
else:
    print("Not enough AF and NSR labeled RR windows to score the real-data baseline in this dataset sample.")


## Real-Data Results Summary

The real-data smoke test successfully read processed ECG parquet and annotation JSON sidecars from MinIO through Spark. It selected `mit-bih-atrial-fibrillation-database`, extracted RR-interval features from clean beat annotations, and labeled 10-second windows using the benchmark AF-vs-NSR policy.

The current run built `1500` labeled windows from `19` recordings: `1187` NSR windows and `313` AF windows. The split is recording-disjoint: train and test have no overlapping `full_recording_id` values. This is the most important correction compared with earlier window-level splits, because it prevents windows from the same recording leaking into both train and test.

Window-level results are the most useful smoke-test signal here. `dummy_prior` behaves like a majority-class baseline: AUROC is `0.5`, sensitivity is `0.0`, and specificity is `1.0`. The `rr_irregularity_rule` is much stronger at the window level: AUROC is about `0.985`, average precision about `0.885`, accuracy about `0.953`, F1 about `0.899`, sensitivity `1.0`, and specificity about `0.94`. This is plausible because AF often produces irregular RR intervals, and this demo uses clean annotated beat positions.

Recording-level results should be treated cautiously. The held-out set contains only `5` recordings, with `4` non-AF recordings and `1` AF-present recording. That makes perfect recording-level metrics for `rr_irregularity_rule` possible but not yet meaningful. The result shows the plumbing works, not that the final benchmark is solved.

Next benchmark steps: generate features for all configured datasets, use the package split manifest rather than notebook sampling, keep subject/recording-safe splits, and evaluate on larger target test sets before drawing model conclusions.


In [ ]:
train_rows[0], train_labels[0]

In [ ]:
from pprint import pprint
import importlib
import importlib.util
import subprocess
import sys

import ecg_benchmark.models.baselines as baselines_mod

baselines_mod = importlib.reload(baselines_mod)
create_model = baselines_mod.create_model

if importlib.util.find_spec("sklearn") is None:
    print("Installing scikit-learn into this notebook kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn>=1.4"])

required_previous_names = ["config", "train_rows", "test_rows", "BinaryPrediction", "compute_binary_metrics", "aggregate_by_recording"]
missing_names = [name for name in required_previous_names if name not in globals()]
if missing_names:
    raise RuntimeError(f"Run the real-data baseline cell above first. Missing: {missing_names}")

train_labels = sorted({int(row["label"]) for row in train_rows})
test_labels = sorted({int(row["label"]) for row in test_rows})
if train_labels != [0, 1] or test_labels != [0, 1]:
    raise RuntimeError(
        "The sklearn baselines need both AF and NSR examples in train and test. "
        f"Got train labels={train_labels}, test labels={test_labels}. Increase MAX_RECORDINGS above."
    )

ml_feature_columns = ["rr_count", "rr_mean", "rr_std", "rr_cv"]
ml_baseline_configs = {
    "logistic_regression_features": {
        "feature_columns": ml_feature_columns,
        "max_iter": 1000,
        "class_weight": "balanced",
        "random_state": config.random_seed,
    },
    "random_forest_features": {
        "feature_columns": ml_feature_columns,
        "n_estimators": 200,
        "max_depth": 4,
        "class_weight": "balanced",
        "random_state": config.random_seed,
    },
}

ml_results = {}
for model_id, model_config in ml_baseline_configs.items():
    model = create_model(model_id).fit(train_rows, [], model_config)
    scores = model.predict_proba(test_rows)
    predictions = [
        BinaryPrediction(y_true=int(row["label"]), y_score=float(score), recording_id=row["full_recording_id"])
        for row, score in zip(test_rows, scores)
    ]
    recording_predictions = aggregate_by_recording(predictions, policy=config.data.recording_aggregation)

    ml_results[model_id] = {
        "feature_columns": getattr(model, "feature_columns", ml_feature_columns),
        "window": compute_binary_metrics(predictions, metric_names=config.metrics),
        "recording": compute_binary_metrics(recording_predictions, metric_names=config.metrics),
    }

for model_id, result in ml_results.items():
    print(f"{model_id} feature columns: {result['feature_columns']}")
    print(f"{model_id} window-level metrics:")
    pprint(result["window"])
    print(f"{model_id} recording-level metrics:")
    pprint(result["recording"])
    print()
